Read FiftyDatasets/AllDatasetsInfo/AllDataSourcesInfo/Kaggle/Sato/ Viznet datasets

In [15]:
import pandas as pd
from datetime import datetime
import re  

DB_or_dataset = input("Insert if using Datasets (d), Database.tables (dt), the two Data Sources (ds), Kaggle Datasets (k), Kaggle Datasets to correct (kc), Sato Datasets (s), Sato Filtered (sf), Filtered Viznet Datasets (v), or Filtered Kaggle Viznet Datasets (kv): ")

print(DB_or_dataset)

if DB_or_dataset.lower() == 'd':
    print("Using Datasets")
    datasets_xlsx = pd.read_excel("FiftyDatasets.xlsx") 
    columns_xlsx = pd.read_excel("AllColumnsFromFiftyDatasets.xlsx")
    analysed_columns_file_path = 'AnalysedColumns.xlsx'     
elif DB_or_dataset.lower() == 'dt':
    print("Using Database tables")
    datasets_xlsx = pd.read_excel("AllDatasetsInfo.xlsx")
    columns_xlsx = pd.read_excel("AllColumnsInfo.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsDB.xlsx'
elif DB_or_dataset.lower() == 'ds':
    print("Using Datasets and Database tables")
    datasets_xlsx = pd.read_excel("AllDataSourcesInfo.xlsx")
    columns_xlsx = pd.read_excel("AllAttributes_andColumnsInfo.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsDS.xlsx'
elif DB_or_dataset.lower() == 'k':
    print("Using Kaggle Datasets")
    datasets_xlsx = pd.read_excel("kaggle_datasets_with_domain_and_match.xlsx")
    columns_xlsx = pd.read_excel("kaggle_headers.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsK.xlsx'
elif DB_or_dataset.lower() == 'kc':
    print("Using Kaggle Datasets Corrections")
    datasets_xlsx = pd.read_excel("kaggle_datasets_with_domain_and_match.xlsx")
    columns_xlsx = pd.read_excel("kaggle_headers_10k.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsKc.xlsx'
elif DB_or_dataset.lower() == 's':
    print("Using Sato Datasets")
    datasets_xlsx = pd.read_excel("datasets_viznet.xlsx")
    columns_xlsx = pd.read_excel("columns_sato_only.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsSato.xlsx'
elif DB_or_dataset.lower() == 'sf':
    print("Using Sato Filtered Datasets")
    datasets_xlsx = pd.read_excel("datasets_viznet.xlsx")
    columns_xlsx = pd.read_excel("filtered_columns_sato_only.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsSatoFiltered.xlsx'
elif DB_or_dataset.lower() == 'v':
    print("Using Viznet Filtered Datasets")
    datasets_xlsx = pd.read_excel("datasets_viznet.xlsx")
    columns_xlsx = pd.read_excel("filtered_columns_viznet_all.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsViznetFiltered.xlsx'
elif DB_or_dataset.lower() == 'kv':
    print("Using Kaggle Viznet Filtered Datasets")
    datasets_xlsx = pd.read_excel("kaggle_viznet_datasets.xlsx")
    columns_xlsx = pd.read_excel("filtered_kaggle_viznet_headers.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsKaggleViznetFiltered.xlsx'
elif DB_or_dataset.lower() == 'sem':
    print("Using SemTabTest 2024 terms")
    datasets_xlsx = pd.read_excel("SemTabTest2024Dataset.xlsx")
    columns_xlsx = pd.read_excel("SemTabTest2024.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsSemTabTest2024.xlsx'
elif DB_or_dataset.lower() == 'sotab':
    print("Using Sotab metadata")
    datasets_xlsx = pd.read_excel("SotabShemasDataset.xlsx")
    columns_xlsx = pd.read_excel("SotabColumns.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsSotab.xlsx'
elif DB_or_dataset.lower() == 't2dv2':
    print("Using t2dv2")
    datasets_xlsx = pd.read_excel("t2dv2Datasets.xlsx")
    columns_xlsx = pd.read_excel("t2dv2Columns.xlsx")
    analysed_columns_file_path = 'AnalysedColumnst2dv2.xlsx'
elif DB_or_dataset.lower() == 'dbp':
    print("Using DbPedia metadata")
    datasets_xlsx = pd.read_excel("DbpediaDataset.xlsx")
    columns_xlsx = pd.read_excel("DbpediaMetadataColumns.xlsx")
    analysed_columns_file_path = 'AnalysedColumnsDbpediaMetadata.xlsx'
else:
    raise ValueError("Invalid input. Please enter 'd', 'dt', 'ds', 'k', 'kc', 's', 'sf', 'v' or 'kv'.")



kv
Using Kaggle Viznet Filtered Datasets


In [16]:
columns_xlsx = columns_xlsx.rename(columns={'index': 'dataset_index'})

columns_xlsx.head()

,dataset_index,name,area,Original Column,ID
0,1001,student-habits-vs-academic-performance,Education,1. student_id,1
1,1001,student-habits-vs-academic-performance,Education,2. age,2
2,1001,student-habits-vs-academic-performance,Education,3. gender,3
3,1001,student-habits-vs-academic-performance,Education,4. study_hours_per_day,4
4,1001,student-habits-vs-academic-performance,Education,5. social_media_hours,5


In [17]:
def normalize_yn_in_original_column(
    df: pd.DataFrame,
    original_col: str = "Original Column",
    flag_col: str = "YN_Header_Flag"
) -> pd.DataFrame:
    """
    Detects 'y/n' or 'Y/N' (with optional spaces around the slash) in the
    ORIGINAL column header and replaces it with 'yes no', so that the existing
    pipeline can pick up 'yes' and infer FinalFormat = binary via formats_dictionary.

    Examples of replacements:
      'Smoker (Y/N)'       -> 'Smoker (yes no)'
      'y/n'                -> 'yes no'
      'Has siblings y / n' -> 'Has siblings yes no'
      'Flag Y/N value'     -> 'Flag yes no value'

    It also writes a boolean flag column to allow later inspection.
    """
    if original_col not in df.columns:
        raise KeyError(f"Column '{original_col}' not found in DataFrame")

    # Regex to find y/n patterns, case-insensitive, with optional spaces
    pattern = re.compile(r'\b[yY]\s*/\s*[nN]\b')

    # Flag rows where the pattern appears
    df[flag_col] = df[original_col].astype(str).str.contains(pattern.pattern, regex=True, na=False)

    # Replace y/n with 'yes no' in the original header
    def _replace_yn(text: str) -> str:
        if not isinstance(text, str):
            return text
        return pattern.sub(" yes no ", text)

    df[original_col] = df[original_col].astype(str).apply(_replace_yn)

    return df


# Example standalone usage
if __name__ == "__main__":
    test_df = pd.DataFrame({
        "Original Column": [
            "1. Smoker (Y/N)",
            "2. y/n",
            "3. Has siblings y / n?",
            "4. Approval (Yes/No)",   # already good, nothing happens
            "5. Random Flag"
        ]
    })

    test_df = normalize_yn_in_original_column(test_df)
    print(test_df)

    print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

             Original Column  YN_Header_Flag
0       1. Smoker ( yes no )            True
1                2.  yes no             True
2  3. Has siblings  yes no ?            True
3       4. Approval (Yes/No)           False
4             5. Random Flag           False
Last run on: 2026-05-10 15:35:59


Clean Columns

In [18]:
import pandas as pd
import re
from datetime import datetime

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Always work on a copy to avoid side effects
    df = df.copy()

    # 0. Normalise Y/N into 'yes no' in the Original Column
    df = normalize_yn_in_original_column(df, original_col="Original Column", flag_col="YN_Header_Flag")

    # 1. Extract the ID from the column name 
    df['ID'] = df['Original Column'].str.extract(r'(\d+)\.')

    # 2. Extract the Column name up to the first colon, slash, or parenthesis
    df['Column'] = df['Original Column'].str.extract(r'\d+\.\s*([^/(:]+)')

    # 3. Extract everything after the first colon, slash, or parenthesis as the full description
    df['Description'] = df['Original Column'].str.extract(r'\d+\.\s*[^/(:]+\s*[:(/](.*)')[0]

    # 4. For cases where Description is None,
    #    extract the content within parentheses
    mask = df['Description'].isnull()
    df.loc[mask, 'Description'] = df.loc[mask, 'Original Column'].str.extract(r'\(([^)]+)\)')

    # 5. Trim spaces from all string columns
    for col in df.columns:
        if df[col].dtype == "object" and col != 'dataset_index':
            df[col] = df[col].str.strip()

    # 6. Replace empty strings with pd.NA  ← changed from None to pd.NA to avoid downcasting warning
    df = df.replace(r'^\s*$', pd.NA, regex=True)
    df = df.infer_objects(copy=False)
    
    # 7. Ensure Description is object dtype before applying .str.replace  ← added to avoid incompatible dtype warning
    df['Description'] = df['Description'].astype(object)

    # 8. Only apply .str.replace on non-null Description rows,
    #    converting to string first to avoid errors
    mask_desc = df['Description'].notnull()
    df.loc[mask_desc, 'Description'] = (
        df.loc[mask_desc, 'Description']
          .astype(str)
          .str.replace(r'\(', ' ', regex=True)
          .str.replace(r'\)', '', regex=True)
    )

    # 9. Reorder columns to ensure ID comes before Column
    columns_order = ['dataset_index', 'name', 'area', 'Original Column', 'ID', 'Column', 'Description']
    columns_order = [col for col in columns_order if col in df.columns]
    df = df[columns_order]

    return df

# Test the function with a sample DataFrame
if __name__ == "__main__":
    test_df = pd.DataFrame({
        'Original Column': [
            '14. num (the predicted attribute)',
            '1. lettr: capital letter (26 values from A to Z)',
            '2. x-box: horizontal position of the box (integer)',
            '1. sepal length in cm',
            '20. Foreign worker (qualitative)',
            '12. A12: Categorical with values: t, f',
            '55. capital_run_length_average (1 continuous real attribute): Average length of uninterrupted sequences of capital letters',
            '1. Class Name (party affiliation): democrat, republican',
            '1.Sex / nominal / -- / M, F, and I (infant)',
            '3.Cell Nucleus 1 - a) radius (mean of distances from center to points on the perimeter)',
            '14. win_loc (varchar(255))',
            '33. #',
            '14.   Rush_Y/A',
            '3.   PCOS (Y/N)'

        ],
        'dataset_index': ['SATO_000001'] * 14,
        'name': [None] * 14,
        'area': ['Sato-Viznet'] * 14
    })

print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
cleaned_df = normalize_yn_in_original_column(test_df)

cleaned_df = clean_columns(cleaned_df)
cleaned_df

Last run on: 2026-05-10 15:35:59


C:\Users\marce\AppData\Local\Temp\ipykernel_38520\3250133709.py:33: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df = df.infer_objects(copy=False)


,dataset_index,name,area,Original Column,ID,Column,Description
0,SATO_000001,None,Sato-Viznet,14. num (the predicted attribute),14,num,the predicted attribute
1,SATO_000001,None,Sato-Viznet,1. lettr: capital letter (26 values from A to Z),1,lettr,capital letter 26 values from A to Z
2,SATO_000001,None,Sato-Viznet,2. x-box: horizontal position of the box (integer),2,x-box,horizontal position of the box integer
3,SATO_000001,None,Sato-Viznet,1. sepal length in cm,1,sepal length in cm,NaN
4,SATO_000001,None,Sato-Viznet,20. Foreign worker (qualitative),20,Foreign worker,qualitative
5,SATO_000001,None,Sato-Viznet,"12. A12: Categorical with values: t, f",12,A12,"Categorical with values: t, f"
6,SATO_000001,None,Sato-Viznet,55. capital_run_length_average (1 continuous real attribute): Average length of uninterrupted sequences of capital letters,55,capital_run_length_average,1 continuous real attribute: Average length of uninterrupted sequences of capital letters
7,SATO_000001,None,Sato-Viznet,"1. Class Name (party affiliation): democrat, republican",1,Class Name,"party affiliation: democrat, republican"
8,SATO_000001,None,Sato-Viznet,"1.Sex / nominal / -- / M, F, and I (infant)",1,Sex,"nominal / -- / M, F, and I infant"
9,SATO_000001,None,Sato-Viznet,3.Cell Nucleus 1 - a) radius (mean of distances from center to points on the perimeter),3,Cell Nucleus 1 - a) radius,mean of distances from center to points on the perimeter


Open formats and abreviations dictionaries

In [19]:
dictionary = {}

# Open the formats dictionary file and read line by line 
with open("formats_dictionary.txt", "r") as file:
    for line in file:
        # Remove the trailing newline and comma, then split the line into key and value at the colon
        key, value = line.rstrip(",\n").split(":")
    
        # Remove the quotes around the key and value
        key = key.strip("'").lower()
        value = value.strip("'")

        # Add the key-value pair to the dictionary
        dictionary[key] = value

target_words_dict = dictionary
description_words_dict = dictionary

# Load the abbreviations dictionary
abbreviations_dict = {}
with open("abbreviations_dictionary.txt", "r") as file:
    for line in file:
        abbr, full_form = line.strip().split(":")
        abbreviations_dict[abbr.strip().lower()] = full_form.strip()
        
print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Last run on: 2026-05-10 15:35:59


Replace abbreviations

In [20]:
def replace_abbreviations(text, abbreviations_dict):
    """
    Tokenises `text` using the original regex and replaces tokens that appear
    in `abbreviations_dict`. In addition, it returns a log of which
    abbreviations were expanded.

    Returns:
        expanded_text, expansion_log

    - expanded_text: the text with abbreviations expanded
    - expansion_log: string like 'gp->games played|bmi->body mass index'
                     or '' (empty string) if nothing was expanded
    """
    if not isinstance(text, str):
        return text, ""

    tokens = re.findall(
        r'\b\w+(?:[-=]\w+)*\b'          # Words, with hyphens/equals (like 0-4, 1=NE)
        r'|[=]'                         # Standalone equal signs
        r'|[°º][CFK]'                   # °C, ºF, etc.
        r'|[°º\-]+'                     # Standalone degree/hyphens
        r'|[a-zA-Z]+/[a-zA-Z]+(?:[²³μµ]?)'   # mg/dL, kg/m², µg/m³, cm², etc.
        r'|\d+[²³]?'                    # 10², 5³ (numbers with superscripts)
        r'|%'                           # <- add this line to capture % as a token
        , text
    )
    #replaced_tokens = [abbreviations_dict.get(token.lower(), token) for token in tokens]
    #return " ".join(replaced_tokens)

    replaced_tokens = []
    used_mappings = []  # e.g. "bmi->body mass index" - we will log "abbr->expansion" here

    for token in tokens:
        key = token.lower()
        if key in abbreviations_dict:
            expansion = abbreviations_dict[key]
            replaced_tokens.append(expansion)
            used_mappings.append(f"{key}->{expansion}")
        else:
            replaced_tokens.append(token)

    # Remove duplicates but keep order
    seen = set()
    unique_used = []
    for m in used_mappings:
        if m not in seen:
            seen.add(m)
            unique_used.append(m)

    expansion_log = "|".join(unique_used)
    expanded_text = " ".join(replaced_tokens)

    return expanded_text, expansion_log

test_df = pd.DataFrame({
    "Original Column": [
        "1. chlorides (mg/dL)",                # plural with unit
        "2. chloride (mg/dL)",                 # singular with unit
        "3. Fuel_Price (USD/L)",               # phrase with underscore, unit
        "4. Tumor-size: 0-4, 5-9, 10-14",      # category ranges with hyphens
        "5. zone: 1=NE, 2=SE, 3=SW, 4=NW",     # code marker with equals
        "6. temperature (°C)",                 # temperature symbol
        "7. temperature (ºF)",                 # alternate degree symbol
        "8. Tdewpoint (from Chievres weather station), °C", # degree symbol at end
        "9. cm² measurement",                  # superscript unit
        "10. workclass",                       # compound word
        "11. landmass",                        # compound word
        "12. thickness (mm³)",                 # unit with superscript ³
        "13. concentration: 5mg/dL",           # value with unit
        "14. value: 10cm³",                    # value with unit
        "15. intensity: 5³, 10²",              # numeric with superscripts
        "16. 0-1 category",                    # hyphen, short
        "17. 1=Red, 2=Blue, 3=Green",          # equals for codes
        "18. percent (%)",                     # symbol in column
        "19. velocity (km/h)",                 # unit with slash
        "20. area: 10cm2, 10cm³",              # unit, no symbol and symbol
        "21. Alcohol (%)",                     # percent
        "22. BMI (kg/m²)",                     # slash, superscript
        "23. air_quality_index (AQI)",         # abbreviation as token
        "24. mean-pH",                         # hyphen in column name
    ]
})

test_df[["AbbrExpanded", "AbbrLog"]] = test_df["Original Column"].apply(
        lambda x: pd.Series(replace_abbreviations(str(x), abbreviations_dict),
                            index=["AbbrExpanded", "AbbrLog"])
    )

pd.set_option("display.max_colwidth", 200)
#print(test_df[["Original Column", "AbbrExpanded", "AbbrLog"]])

test_df[['Original Column', "AbbrExpanded", "AbbrLog"]]

# Apply the abbreviation/tokenizer function to every row
#test_df['AbbrExpanded'] = test_df['Original Column'].apply(lambda x: replace_abbreviations(str(x), abbreviations_dict))

# Display original and processed columns side by side for easy comparison
#pd.set_option('display.max_colwidth', 200)
#test_df[['Original Column', 'AbbrExpanded']]

#print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


,Original Column,AbbrExpanded,AbbrLog
0,1. chlorides (mg/dL),1 chlorides milligrams dL,mg->milligrams
1,2. chloride (mg/dL),2 chloride milligrams dL,mg->milligrams
2,3. Fuel_Price (USD/L),3 Fuel_Price USD L,
3,"4. Tumor-size: 0-4, 5-9, 10-14",4 Tumor-size 0-4 5-9 10-14,
4,"5. zone: 1=NE, 2=SE, 3=SW, 4=NW",5 zone 1=NE 2=SE 3=SW 4=NW,
5,6. temperature (°C),6 temperature °C,
6,7. temperature (ºF),7 temperature ºF,
7,"8. Tdewpoint (from Chievres weather station), °C",8 Tdewpoint from Chievres weather station °C,
8,9. cm² measurement,9 cm² measurement,
9,10. workclass,10 workclass,


Split Camel Case

In [21]:
import re

def split_camel_case(s):
    if s.lower() == 'ph':
        return 'ph'
    # Split on separators
    tokens = re.split(r'[\s._\-]+', s)
    processed_tokens = []

    for token in tokens:
        if not token:
            continue
        # If all uppercase or digits, keep as is
        if token.isupper() or token.isdigit():
            processed_tokens.append(token.lower())
            continue
        # If all-uppercase+digits, keep as is
        if re.match(r'^[A-Z]{2,}\d+$', token):
            processed_tokens.append(token.lower())
            continue

        # Character-by-character scan
        words = []
        current = ''
        i = 0
        while i < len(token):
            c = token[i]
            if current == '':
                current = c
            elif (
                # lower->upper transition
                (current[-1].islower() and c.isupper()) or
                # digit->letter or letter->digit
                (current[-1].isdigit() and c.isalpha()) or
                (current[-1].isalpha() and c.isdigit())
            ):
                words.append(current)
                current = c
            else:
                current += c
            i += 1
        if current:
            words.append(current)

        # Now check if the last two or more are all caps: join them!
        if len(words) >= 2 and all(w.isupper() for w in words[-2:]) and len(''.join(words[-2:])) >= 2:
            # Join last all-cap runs
            n = len(words) - 1
            while n > 0 and words[n].isupper():
                n -= 1
            # All uppercase words from n+1 to end are the tail
            head = words[:n+1]
            tail = ''.join(words[n+1:])
            processed_tokens.extend([w.lower() for w in head if w])
            if tail:
                processed_tokens.append(tail.lower())
        else:
            processed_tokens.extend([w.lower() for w in words if w])

    return ' '.join(processed_tokens)

print(split_camel_case("reviews.sourceURLs"))   # reviews source urls
print(split_camel_case("prices.sourceURLs"))    # prices source urls
print(split_camel_case("sourceURLs"))           # source urls
print(split_camel_case("UrlLength"))            # url length
print(split_camel_case("nation_flag_url"))      # nation flag url
print(split_camel_case("URLs"))              # mean ph
print(split_camel_case("JSONFileURL"))          # json file url
print(split_camel_case("ODIDs"))                # odids
print(split_camel_case("ODID"))                # odid


reviews source urls
prices source urls
source urls
url length
nation flag url
urls
jsonfile url
odids
odid


In [22]:
columns_xlsx

,dataset_index,name,area,Original Column,ID
0,1001,student-habits-vs-academic-performance,Education,1. student_id,1
1,1001,student-habits-vs-academic-performance,Education,2. age,2
2,1001,student-habits-vs-academic-performance,Education,3. gender,3
3,1001,student-habits-vs-academic-performance,Education,4. study_hours_per_day,4
4,1001,student-habits-vs-academic-performance,Education,5. social_media_hours,5
...,...,...,...,...,...
118634,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,2. Name,2
118635,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,3. Friendly,3
118636,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,4. Honored,4
118637,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,5. Revered,5


In [23]:
cleaned_columns_df = clean_columns(columns_xlsx.copy())
cleaned_columns_df

C:\Users\marce\AppData\Local\Temp\ipykernel_38520\3250133709.py:33: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df = df.infer_objects(copy=False)


,dataset_index,name,area,Original Column,ID,Column,Description
0,1001,student-habits-vs-academic-performance,Education,1. student_id,1,student_id,NaN
1,1001,student-habits-vs-academic-performance,Education,2. age,2,age,NaN
2,1001,student-habits-vs-academic-performance,Education,3. gender,3,gender,NaN
3,1001,student-habits-vs-academic-performance,Education,4. study_hours_per_day,4,study_hours_per_day,NaN
4,1001,student-habits-vs-academic-performance,Education,5. social_media_hours,5,social_media_hours,NaN
...,...,...,...,...,...,...,...
118634,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,2. Name,2,Name,NaN
118635,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,3. Friendly,3,Friendly,NaN
118636,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,4. Honored,4,Honored,NaN
118637,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,5. Revered,5,Revered,NaN


Preprocess Columns

In [24]:
def preprocess_columns(df):
        
    """
    - Creates CleanedColumn from Column (split_camel_case + cleaning).
    - Applies replace_abbreviations to both CleanedColumn and Description.
    - Stores mapping logs in:
        df['abbrev_clean']  (from CleanedColumn)
        df['abbrev_desc']   (from Description)
    """

    # Only convert "Column" to string type for non-null values and then to lowercase to create "CleanedColumn"

    
    mask_col = df['Column'].notna()

    df.loc[mask_col, 'Column'] = df.loc[mask_col, 'Column'].str.replace('#', 'number', regex=False)
    df.loc[mask_col, 'Column'] = df.loc[mask_col, 'Column'].str.replace('$', 'dollar', regex=False)

    df.loc[mask_col, 'CleanedColumn'] = df.loc[mask_col, 'Column'].astype(str).apply(split_camel_case)
    
    # Remove '-' and '_' characters only for non-null values
    mask_clean = df['CleanedColumn'].notna()
    df.loc[mask_clean, 'CleanedColumn'] = df.loc[mask_clean, 'CleanedColumn'].str.replace('[-_]', ' ', regex=True)
    
    # Ensure log columns exist
    if 'abbrev_clean' not in df.columns:
        df['abbrev_clean'] = pd.NA
    if 'abbrev_desc' not in df.columns:
        df['abbrev_desc'] = pd.NA

    if mask_clean.any():
        expanded_clean = df.loc[mask_clean, 'CleanedColumn'].apply(
            lambda x: pd.Series(
                replace_abbreviations(x, abbreviations_dict),
                index=['CleanedColumn', 'abbrev_clean_tmp']
            )
        )
        df.loc[mask_clean, 'CleanedColumn'] = expanded_clean['CleanedColumn']
        df.loc[mask_clean, 'abbrev_clean'] = expanded_clean['abbrev_clean_tmp']

    mask_desc = df['Description'].notna()
    if mask_desc.any():
        expanded_desc = df.loc[mask_desc, 'Description'].apply(
            lambda x: pd.Series(
                replace_abbreviations(x, abbreviations_dict),
                index=['Description', 'abbrev_desc_tmp']
            )
        )
        df.loc[mask_desc, 'Description'] = expanded_desc['Description']
        df.loc[mask_desc, 'abbrev_desc'] = expanded_desc['abbrev_desc_tmp']     

    return df

# Clean the columns

cleaned_columns_df = clean_columns(columns_xlsx.copy())

# Preprocess the column names
preprocessed_columns_df = preprocess_columns(cleaned_columns_df)
print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

preprocessed_columns_df



C:\Users\marce\AppData\Local\Temp\ipykernel_38520\3250133709.py:33: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df = df.infer_objects(copy=False)


Last run on: 2026-05-10 15:37:46


,dataset_index,name,area,Original Column,ID,Column,Description,CleanedColumn,abbrev_clean,abbrev_desc
0,1001,student-habits-vs-academic-performance,Education,1. student_id,1,student_id,NaN,student id,,<NA>
1,1001,student-habits-vs-academic-performance,Education,2. age,2,age,NaN,age,,<NA>
2,1001,student-habits-vs-academic-performance,Education,3. gender,3,gender,NaN,gender,,<NA>
3,1001,student-habits-vs-academic-performance,Education,4. study_hours_per_day,4,study_hours_per_day,NaN,study hours per day,,<NA>
4,1001,student-habits-vs-academic-performance,Education,5. social_media_hours,5,social_media_hours,NaN,social media hours,,<NA>
...,...,...,...,...,...,...,...,...,...,...
118634,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,2. Name,2,Name,NaN,name,,<NA>
118635,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,3. Friendly,3,Friendly,NaN,friendly,,<NA>
118636,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,4. Honored,4,Honored,NaN,honored,,<NA>
118637,SATO_079984,0_1438042987171.38;warc;CC-MAIN-20150728002307-00339-ip-10-236-191-2.ec2.internal.json.gz_716-Southsea Freebooters - WoW_HKCEY7NCAYUBBODC,Sato-Viznet,5. Revered,5,Revered,NaN,revered,,<NA>


KG Matching

In [25]:
import pandas as pd
import difflib
import numpy as np
from gensim.models.fasttext import load_facebook_vectors
from datetime import datetime


# Load DBpedia and Schema.org ontology terms from your provided files
dbpedia_terms_df = pd.read_excel("dbpedia.xlsx")
dbpedia_labels = dbpedia_terms_df["cleaned_label"].astype(str).str.lower().str.strip().tolist()
dbpedia_ids = dbpedia_terms_df["id"].astype(str).tolist()
dbpedia_label_to_id = dict(zip(dbpedia_labels, dbpedia_ids))

schema_terms_df = pd.read_excel("schema.xlsx")
schema_labels = schema_terms_df["cleaned_label"].astype(str).str.lower().str.strip().tolist()
schema_ids = schema_terms_df["id"].astype(str).tolist()
schema_label_to_id = dict(zip(schema_labels, schema_ids))

print("Loaded DBpedia labels:", dbpedia_labels[:5], "...")
print("Loaded Schema.org labels:", schema_labels[:5], "...")


def syntactic_match(header, labels_list, label_to_id, threshold=0.9):
    """
    Match a header against ontology labels using exact matching first,
    then difflib close matching.

    Parameters
    ----------
    header : str
        Header text to match.
    labels_list : list
        List of ontology labels.
    label_to_id : dict
        Dictionary mapping ontology labels to ontology IDs.
    threshold : float
        Cutoff for difflib close matching.

    Returns
    -------
    str or None
        Matched ontology ID, or None if no match is found.
    """
    header = str(header).strip().lower()

    if header in label_to_id:
        return label_to_id[header]

    matches = difflib.get_close_matches(header, labels_list, n=1, cutoff=threshold)

    if matches:
        return label_to_id[matches[0]]

    return None


# Load Gensim FastText model
fasttext_model = load_facebook_vectors("cc.en.300.bin")

print("Gensim FastText model loaded. Dimension:", fasttext_model.vector_size)


def get_embedding(text):
    """
    Return the average FastText embedding for a text string.

    Parameters
    ----------
    text : str
        Text to embed.

    Returns
    -------
    numpy.ndarray
        Average embedding vector. If no valid token is found, returns a zero vector.
    """
    if text is None:
        return np.zeros(fasttext_model.vector_size, dtype=np.float32)

    text = str(text).strip().lower()

    if not text:
        return np.zeros(fasttext_model.vector_size, dtype=np.float32)

    words = text.split()
    vectors = []

    for word in words:
        try:
            vectors.append(fasttext_model.get_vector(word))
        except KeyError:
            continue

    if not vectors:
        return np.zeros(fasttext_model.vector_size, dtype=np.float32)

    return np.mean(vectors, axis=0).astype(np.float32)


def normalise_vector(vector):
    """
    Return a unit-normalised vector.

    Parameters
    ----------
    vector : numpy.ndarray
        Input vector.

    Returns
    -------
    numpy.ndarray
        Unit-normalised vector. If the norm is zero, returns the original vector.
    """
    norm = np.linalg.norm(vector)

    if norm == 0:
        return vector

    return vector / norm


def build_label_embedding_matrix(labels_list, label_to_id):
    """
    Build sorted labels, IDs, and a normalised embedding matrix for ontology labels.

    Parameters
    ----------
    labels_list : list
        List of ontology labels.
    label_to_id : dict
        Dictionary mapping ontology labels to ontology IDs.

    Returns
    -------
    tuple
        sorted_labels, labels_array, ids_array, embeddings_matrix
    """
    cleaned_labels = []
    cleaned_ids = []
    cleaned_embeddings = []

    unique_labels = sorted(set(str(label).strip().lower() for label in labels_list if str(label).strip()))

    for label in unique_labels:
        if label not in label_to_id:
            continue

        embedding = get_embedding(label)
        embedding = normalise_vector(embedding)

        cleaned_labels.append(label)
        cleaned_ids.append(label_to_id[label])
        cleaned_embeddings.append(embedding)

    labels_array = np.array(cleaned_labels, dtype=object)
    ids_array = np.array(cleaned_ids, dtype=object)
    embeddings_matrix = np.vstack(cleaned_embeddings).astype(np.float32)

    sorted_labels = sorted(
        cleaned_labels,
        key=lambda value: (len(value.split()), len(value)),
        reverse=True
    )

    return sorted_labels, labels_array, ids_array, embeddings_matrix


def semantic_match_fast(header, labels_array, ids_array, embeddings_matrix, similarity_threshold=0.75):
    """
    Find the best semantic match using vectorised cosine similarity.

    Parameters
    ----------
    header : str
        Header text.
    labels_array : numpy.ndarray
        Ontology labels aligned with ids_array and embeddings_matrix.
    ids_array : numpy.ndarray
        Ontology IDs aligned with labels_array and embeddings_matrix.
    embeddings_matrix : numpy.ndarray
        Normalised ontology label embeddings.
    similarity_threshold : float
        Minimum similarity score required to accept the semantic match.

    Returns
    -------
    tuple
        ontology_id, similarity_score
    """
    header_embedding = get_embedding(header)
    header_embedding = normalise_vector(header_embedding)

    if np.linalg.norm(header_embedding) == 0:
        return None, 0.0

    similarities = embeddings_matrix @ header_embedding

    best_index = int(np.argmax(similarities))
    best_score = float(similarities[best_index])

    if best_score >= similarity_threshold:
        return ids_array[best_index], best_score

    return None, best_score


dbpedia_sorted_labels, dbpedia_labels_array, dbpedia_ids_array, dbpedia_embeddings_matrix = build_label_embedding_matrix(
    dbpedia_labels,
    dbpedia_label_to_id
)

schema_sorted_labels, schema_labels_array, schema_ids_array, schema_embeddings_matrix = build_label_embedding_matrix(
    schema_labels,
    schema_label_to_id
)

print("DBpedia embedding matrix:", dbpedia_embeddings_matrix.shape)
print("Schema.org embedding matrix:", schema_embeddings_matrix.shape)

print("Embedding for 'birth date':", get_embedding("birth date")[:5])

print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Loaded DBpedia labels: ['a side', 'abbeychurch blessing', 'abbeychurch blessing charge', 'abbreviation', 'able to grind'] ...
Loaded Schema.org labels: ['3 d model', 'abdomen', 'about', 'about page', 'abridged'] ...
Gensim FastText model loaded. Dimension: 300
DBpedia embedding matrix: (3036, 300)
Schema.org embedding matrix: (2637, 300)
Embedding for 'birth date': [ 0.00283275  0.0443643  -0.06757186  0.06091432  0.00819952]
Last run on: 2026-05-10 15:40:22


Apply analysis

In [26]:
import pandas as pd
import re
from datetime import datetime


def match_with_plural(token, dictionary):
    """
    Match a token against a dictionary, including simple plural-to-singular variants.

    Parameters
    ----------
    token : str
        Token to be matched.
    dictionary : dict
        Dictionary containing target words and their formats.

    Returns
    -------
    str or None
        Matched format, or None if no match is found.
    """
    if token in dictionary:
        return dictionary[token]

    if token.endswith("ies") and len(token) > 3:
        singular = token[:-3] + "y"
        if singular in dictionary:
            return dictionary[singular]

    if token.endswith("es") and len(token) > 2:
        singular = token[:-2]
        if singular in dictionary:
            return dictionary[singular]

    if token.endswith("s") and len(token) > 2:
        singular = token[:-1]
        if singular in dictionary:
            return dictionary[singular]

    return None


def apply_analysis(df, target_words_dict, description_words_dict, abbreviations_dict, all_datasets_info):
    """
    Apply FinalFormat analysis to cleaned column names and descriptions.

    Parameters
    ----------
    df : pandas.DataFrame
        Preprocessed columns DataFrame.
    target_words_dict : dict
        Dictionary mapping column-label terms to FinalFormat values.
    description_words_dict : dict
        Dictionary mapping description terms to FinalFormat values.
    abbreviations_dict : dict
        Dictionary mapping abbreviations to expanded terms.
    all_datasets_info : pandas.DataFrame
        DataFrame containing dataset or table metadata.

    Returns
    -------
    pandas.DataFrame
        DataFrame with ColumnKeyword, ColumnFormat, DescriptionKeyword, and DescriptionFormat.
    """

    def get_special_name_format(table_name, column_name):
        """
        Handle special cases where a generic 'name' column should inherit context from the table name.

        Parameters
        ----------
        table_name : str
            Dataset or table name.
        column_name : str
            Original column name.

        Returns
        -------
        tuple
            keyword, format_type
        """
        special_cases = {
            "city": "city",
            "country": "country",
            "state": "state",
            "province": "state"
        }

        if DB_or_dataset.lower() == "d":
            if column_name.lower() == "name":
                return "name", target_words_dict["name"]
        else:
            table_name_str = str(table_name).lower()

            for case, format_type in special_cases.items():
                if case in table_name_str and column_name.lower() == "name":
                    return f"{format_type}_name", target_words_dict[format_type]

        return "name", target_words_dict["name"]

    has_pk_fk_info = (
        "primary_key" in all_datasets_info.columns and
        "foreign_keys" in all_datasets_info.columns
    )

    pk_fk_dict = {}

    if has_pk_fk_info:
        for _, row in all_datasets_info.iterrows():
            table_name = row["index"]
            pk = row["primary_key"] if pd.notna(row["primary_key"]) else None
            fks = row["foreign_keys"].split(", ") if pd.notna(row["foreign_keys"]) else []
            fk_columns = [fk.split(" -> ")[0] for fk in fks]
            pk_fk_dict[table_name] = {"pk": pk, "fks": fk_columns}

    formats_ordered_list = list(target_words_dict.keys())

    df["ColumnKeyword"] = None
    df["ColumnFormat"] = None
    df["DescriptionKeyword"] = None
    df["DescriptionFormat"] = None

    count = 0

    for i, row in df.iterrows():
        count += 1

        if count % 1000 == 0:
            print(f"✅ FinalFormat column analysis: processed {count:,} rows...")

        std_col_name = row["CleanedColumn"]
        col_name = row["Column"]
        table_name = row["dataset_index"]

        if pd.isnull(std_col_name):
            continue

        std_col_name = split_camel_case(std_col_name)

        if table_name in pk_fk_dict:
            if col_name == pk_fk_dict[table_name]["pk"] or col_name in pk_fk_dict[table_name]["fks"]:
                df.at[i, "ColumnKeyword"] = "id"
                df.at[i, "ColumnFormat"] = "IDcolumn"
                continue

        found = False

        for word in formats_ordered_list:
            analysis = target_words_dict[word.lower()]

            if word == "name":
                keyword, format_type = get_special_name_format(table_name, col_name)

                if keyword != "name":
                    df.at[i, "ColumnKeyword"] = keyword
                    df.at[i, "ColumnFormat"] = format_type
                    found = True
                    break

                pattern = rf"({word})(?![\w-])"

            elif word == "id" and str(col_name).endswith("ID"):
                df.at[i, "ColumnKeyword"] = "id"
                df.at[i, "ColumnFormat"] = target_words_dict.get("id", "ID column")
                found = True
                break

            else:
                pattern = rf"\b{word}\b"

            if re.search(pattern, std_col_name, re.IGNORECASE):
                df.at[i, "ColumnKeyword"] = word
                df.at[i, "ColumnFormat"] = analysis
                found = True
                break

        if not found:
            replaced_text, abbr_log = replace_abbreviations(std_col_name, abbreviations_dict)

            if abbr_log:
                df.at[i, "abbrev_clean_analysis"] = abbr_log

            for word in formats_ordered_list:
                pattern = rf"\b{word}\b"

                if re.search(pattern, replaced_text, re.IGNORECASE):
                    df.at[i, "ColumnKeyword"] = word
                    df.at[i, "ColumnFormat"] = target_words_dict[word.lower()]
                    found = True
                    break

        if not found:
            tokens = re.findall(r"\b\w+\b|%", row["CleanedColumn"].lower())

            for token in tokens:
                expanded_token = abbreviations_dict.get(token.lower(), None)

                if expanded_token:
                    match = match_with_plural(expanded_token, target_words_dict)

                    if match:
                        df.at[i, "ColumnKeyword"] = expanded_token
                        df.at[i, "ColumnFormat"] = match
                        found = True
                        break

                match = match_with_plural(token, target_words_dict)

                if match:
                    df.at[i, "ColumnKeyword"] = token
                    df.at[i, "ColumnFormat"] = match
                    found = True
                    break

        tokens = re.findall(r"\b\w+\b|%", row["CleanedColumn"].lower())

        if "%" in tokens and "%" in target_words_dict:
            df.at[i, "ColumnKeyword"] = "%"
            df.at[i, "ColumnFormat"] = target_words_dict["%"]

    count = 0

    for i, row in df.iterrows():
        count += 1

        if count % 1000 == 0:
            print(f"✅ FinalFormat description analysis: processed {count:,} rows...")

        if pd.isnull(row["Description"]):
            continue

        for word, analysis in description_words_dict.items():
            if word in ["is", "has"]:
                pattern = r"^\s*" + re.escape(word) + r"\b"

                if re.search(pattern, row["Description"], re.IGNORECASE):
                    df.at[i, "DescriptionKeyword"] = word
                    df.at[i, "DescriptionFormat"] = analysis
                    break

            else:
                if re.search(rf"\b{re.escape(word)}\b", row["Description"], re.IGNORECASE):
                    df.at[i, "DescriptionKeyword"] = word
                    df.at[i, "DescriptionFormat"] = analysis
                    break

                elif not word.isalnum():
                    if word.lower() in row["Description"].lower():
                        df.at[i, "DescriptionKeyword"] = word
                        df.at[i, "DescriptionFormat"] = analysis
                        break

    return df


analysed_columns_df = apply_analysis(
    preprocessed_columns_df.copy(),
    target_words_dict,
    description_words_dict,
    abbreviations_dict,
    datasets_xlsx
)


def match_with_ontology_fast(
    header,
    labels_list,
    sorted_labels,
    label_to_id,
    labels_array,
    ids_array,
    embeddings_matrix,
    syntactic_thresh=1.0,
    semantic_thresh=0.75,
    column_keyword=None
):
    """
    Match a header against one ontology using four stages:
    1. syntactic match on the full header;
    2. substring match on ontology labels contained in the header;
    3. vectorised semantic match on the full header;
    4. syntactic and semantic fallback using ColumnKeyword.

    Parameters
    ----------
    header : str
        Cleaned column header.
    labels_list : list
        List of ontology labels.
    sorted_labels : list
        Ontology labels sorted by length for substring preference.
    label_to_id : dict
        Dictionary mapping ontology labels to ontology IDs.
    labels_array : numpy.ndarray
        Ontology labels aligned with ids_array and embeddings_matrix.
    ids_array : numpy.ndarray
        Ontology IDs aligned with labels_array and embeddings_matrix.
    embeddings_matrix : numpy.ndarray
        Normalised ontology label embeddings.
    syntactic_thresh : float
        Threshold for syntactic close matching.
    semantic_thresh : float
        Threshold for semantic matching.
    column_keyword : str or None
        Source keyword identified by the FinalFormat logic.

    Returns
    -------
    tuple
        ontology_id, score
    """
    header_lc = str(header).lower().strip()

    if not header_lc or header_lc == "nan":
        return "Nil", 0.0

    match_id = syntactic_match(
        header_lc,
        labels_list,
        label_to_id,
        threshold=syntactic_thresh
    )

    if match_id:
        return match_id, 1.0

    for label in sorted_labels:
        if len(label) >= 4 and label in header_lc:
            return label_to_id[label], 1.0

    match_id, sim_score = semantic_match_fast(
        header_lc,
        labels_array,
        ids_array,
        embeddings_matrix,
        similarity_threshold=semantic_thresh
    )

    if match_id:
        return match_id, sim_score

    if column_keyword is not None and not pd.isna(column_keyword):
        keyword_lc = str(column_keyword).lower().strip()

        if keyword_lc and keyword_lc != "nan":
            match_id = syntactic_match(
                keyword_lc,
                labels_list,
                label_to_id,
                threshold=syntactic_thresh
            )

            if match_id:
                return match_id, 1.0

            match_id, keyword_score = semantic_match_fast(
                keyword_lc,
                labels_array,
                ids_array,
                embeddings_matrix,
                similarity_threshold=semantic_thresh
            )

            if match_id:
                return match_id, keyword_score

    return "Nil", 0.0


def build_kg_cache(
    unique_pairs_df,
    kg_name,
    labels_list,
    sorted_labels,
    label_to_id,
    labels_array,
    ids_array,
    embeddings_matrix,
    syntactic_thresh=1.0,
    semantic_thresh=0.75
):
    """
    Build a cache of KG matches for unique CleanedColumn and ColumnKeyword pairs.

    Parameters
    ----------
    unique_pairs_df : pandas.DataFrame
        DataFrame containing unique CleanedColumn and ColumnKeyword pairs.
    kg_name : str
        Prefix for output columns, for example 'DBpedia' or 'Schema'.
    labels_list : list
        List of ontology labels.
    sorted_labels : list
        Ontology labels sorted by length for substring preference.
    label_to_id : dict
        Dictionary mapping ontology labels to ontology IDs.
    labels_array : numpy.ndarray
        Ontology labels aligned with ids_array and embeddings_matrix.
    ids_array : numpy.ndarray
        Ontology IDs aligned with labels_array and embeddings_matrix.
    embeddings_matrix : numpy.ndarray
        Normalised ontology label embeddings.
    syntactic_thresh : float
        Threshold for syntactic close matching.
    semantic_thresh : float
        Threshold for semantic matching.

    Returns
    -------
    pandas.DataFrame
        Cache DataFrame with KG type and score for each unique pair.
    """
    cache_rows = []
    total = len(unique_pairs_df)

    for position, row in enumerate(unique_pairs_df.itertuples(index=False), start=1):
        if position % 1000 == 0:
            print(f"✅ {kg_name}: processed {position:,} unique header-keyword pairs out of {total:,}...")

        header = getattr(row, "CleanedColumn")
        column_keyword = getattr(row, "ColumnKeyword")

        kg_type, kg_score = match_with_ontology_fast(
            header=header,
            labels_list=labels_list,
            sorted_labels=sorted_labels,
            label_to_id=label_to_id,
            labels_array=labels_array,
            ids_array=ids_array,
            embeddings_matrix=embeddings_matrix,
            syntactic_thresh=syntactic_thresh,
            semantic_thresh=semantic_thresh,
            column_keyword=column_keyword
        )

        cache_rows.append(
            {
                "CleanedColumn": header,
                "ColumnKeyword": column_keyword,
                f"{kg_name}Type": kg_type,
                f"{kg_name}Score": kg_score
            }
        )

    return pd.DataFrame(cache_rows)


def annotate_all_kgs_fast(df, syntactic_thresh=1.0, semantic_thresh=0.75):
    """
    Annotate all rows with DBpedia and Schema.org types using cached unique pairs.

    Parameters
    ----------
    df : pandas.DataFrame
        Analysed columns DataFrame.
    syntactic_thresh : float
        Threshold for syntactic close matching.
    semantic_thresh : float
        Threshold for semantic matching.

    Returns
    -------
    pandas.DataFrame
        DataFrame with DBpediaType, DBpediaScore, SchemaOrgType, and SchemaScore.
    """
    df = df.copy()

    if "ColumnKeyword" not in df.columns:
        df["ColumnKeyword"] = None

    df["CleanedColumn"] = df["CleanedColumn"].fillna("").astype(str).str.lower().str.strip()
    df["ColumnKeyword"] = df["ColumnKeyword"].where(df["ColumnKeyword"].notna(), "")

    unique_pairs_df = df[["CleanedColumn", "ColumnKeyword"]].drop_duplicates().copy()

    print("Total rows:", len(df))
    print("Unique CleanedColumn + ColumnKeyword pairs:", len(unique_pairs_df))

    dbpedia_cache_df = build_kg_cache(
        unique_pairs_df=unique_pairs_df,
        kg_name="DBpedia",
        labels_list=dbpedia_labels,
        sorted_labels=dbpedia_sorted_labels,
        label_to_id=dbpedia_label_to_id,
        labels_array=dbpedia_labels_array,
        ids_array=dbpedia_ids_array,
        embeddings_matrix=dbpedia_embeddings_matrix,
        syntactic_thresh=syntactic_thresh,
        semantic_thresh=semantic_thresh
    )

    schema_cache_df = build_kg_cache(
        unique_pairs_df=unique_pairs_df,
        kg_name="Schema",
        labels_list=schema_labels,
        sorted_labels=schema_sorted_labels,
        label_to_id=schema_label_to_id,
        labels_array=schema_labels_array,
        ids_array=schema_ids_array,
        embeddings_matrix=schema_embeddings_matrix,
        syntactic_thresh=syntactic_thresh,
        semantic_thresh=semantic_thresh
    )

    df = df.merge(
        dbpedia_cache_df,
        on=["CleanedColumn", "ColumnKeyword"],
        how="left"
    )

    df = df.merge(
        schema_cache_df,
        on=["CleanedColumn", "ColumnKeyword"],
        how="left"
    )

    df = df.rename(columns={"SchemaType": "SchemaOrgType"})

    return df


analysed_columns_df["ColumnDescCombined"] = (
    analysed_columns_df["CleanedColumn"].fillna("").astype(str) + " " +
    analysed_columns_df["Description"].fillna("").astype(str)
)

analysed_columns_df = annotate_all_kgs_fast(
    analysed_columns_df,
    syntactic_thresh=1.0,
    semantic_thresh=0.75
)

ac = analysed_columns_df[
    [
        "CleanedColumn",
        "Description",
        "DBpediaType",
        "DBpediaScore",
        "SchemaOrgType",
        "SchemaScore"
    ]
].head(60)

print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ac\



✅ FinalFormat column analysis: processed 1,000 rows...
✅ FinalFormat column analysis: processed 2,000 rows...
✅ FinalFormat column analysis: processed 3,000 rows...
✅ FinalFormat column analysis: processed 4,000 rows...
✅ FinalFormat column analysis: processed 5,000 rows...
✅ FinalFormat column analysis: processed 6,000 rows...
✅ FinalFormat column analysis: processed 7,000 rows...
✅ FinalFormat column analysis: processed 8,000 rows...
✅ FinalFormat column analysis: processed 9,000 rows...
✅ FinalFormat column analysis: processed 10,000 rows...
✅ FinalFormat column analysis: processed 11,000 rows...
✅ FinalFormat column analysis: processed 12,000 rows...
✅ FinalFormat column analysis: processed 13,000 rows...
✅ FinalFormat column analysis: processed 14,000 rows...
✅ FinalFormat column analysis: processed 15,000 rows...
✅ FinalFormat column analysis: processed 16,000 rows...
✅ FinalFormat column analysis: processed 17,000 rows...
✅ FinalFormat column analysis: processed 18,000 rows...
✅

,CleanedColumn,Description,DBpediaType,DBpediaScore,SchemaOrgType,SchemaScore
0,student id,NaN,http://dbpedia.org/ontology/student,1.000000,https://schema.org/identifier,0.990018
1,age,NaN,http://dbpedia.org/ontology/age,1.000000,https://schema.org/suggestedAge,0.980323
2,gender,NaN,http://dbpedia.org/ontology/gender,1.000000,https://schema.org/gender,1.000000
3,study hours per day,NaN,http://dbpedia.org/ontology/passengersPerDay,0.937115,https://schema.org/study,1.000000
4,social media hours,NaN,http://dbpedia.org/ontology/media,1.000000,https://schema.org/SocialMediaPosting,0.849450
5,netflix hours,NaN,http://dbpedia.org/ontology/flyingHours,0.791533,https://schema.org/hoursAvailable,0.751610
6,part time job,NaN,http://dbpedia.org/ontology/part,1.000000,https://schema.org/Time,1.000000
7,attendance percentage,NaN,http://dbpedia.org/ontology/percentage,1.000000,Nil,0.000000
8,sleep hours,NaN,http://dbpedia.org/ontology/flyingHours,0.791533,https://schema.org/hoursAvailable,0.925501
9,diet quality,NaN,Nil,0.000000,https://schema.org/diet,1.000000


In [27]:
header = "au release date"
matches = [label for label in dbpedia_labels if (label in header or header in label) and len(label) >= 4]
matches = sorted(matches, key=lambda x: (len(x.split()), len(x)), reverse=True)
print(matches)

['release date', 'date']


FinalFormat

In [28]:
def analysis_of_column(ColumnFormat, DescriptionFormat, DescriptionKeyword, index_value):
    if ColumnFormat is None and DescriptionFormat is None:
        return 'NaN'

    # If either is 'None', choose the one that has a value
    if pd.isnull(ColumnFormat):
        return DescriptionFormat
    elif pd.isnull(DescriptionFormat):
        return ColumnFormat
  
    # Prioritize 'categorical' over 'numerical', but prefer specific numeric types in DescriptionKeyword
    if ColumnFormat == 'categorical' and DescriptionFormat == 'numerical':
        if DescriptionKeyword in ['float', 'double']:
            return DescriptionFormat
        return ColumnFormat
    
    # If one is 'string' and the other is not, choose the one that is not 'string'
    if ColumnFormat == 'string' and DescriptionFormat not in [None, 'string', 'NaN']:
        return DescriptionFormat
    elif DescriptionFormat == 'string' and ColumnFormat not in [None, 'string', 'NaN']:
        return ColumnFormat

    # If one is 'hour' and the other is numerical, choose numerical'
    if ColumnFormat == 'hour' and DescriptionFormat.startswith("numerical"):
        return DescriptionFormat
    elif DescriptionFormat == 'hour' and ColumnFormat.startswith("numerical"):
        return ColumnFormat
    
    # Datetime should always win
    if ColumnFormat == 'datetime' or DescriptionFormat == 'datetime':
        return 'datetime'

    # ID column should always win
    if ColumnFormat == "IDcolumn" or DescriptionFormat == "IDcolumn":
        return "IDcolumn"
    
    # percentage should always win
    if ColumnFormat == "percentage" or DescriptionFormat == "percentage":
        return "percentage"
    
    # binary should always win
    if ColumnFormat == 'binary' or DescriptionFormat == 'binary':
        return 'binary' 

    # If both are 'numerical' or if ColumnFormat is in the predefined list, return ColumnFormat
    if (ColumnFormat.startswith("numerical") and DescriptionFormat.startswith("numerical")) or \
       ColumnFormat in ["phone", "month", "date", "weekday", "week", "year",
                        "country", "state", "city", "street", "name",
                        "latitude", "longitude", "postalcode", "URLformat","IPformat", 
                        "E-mailformat", "binary"]:
        return ColumnFormat
    elif ColumnFormat in ["age"] and DescriptionFormat not in ['categorical']:
        return ColumnFormat
    # If no exceptions apply, return the value from "DescriptionFormat"
    else:
        return DescriptionFormat

# Apply the function to create the new column
analysed_columns_df['FinalFormat'] = analysed_columns_df.apply(
    lambda row: analysis_of_column(
        row['ColumnFormat'],
        row['DescriptionFormat'],
        row['DescriptionKeyword'],
        row['dataset_index']),  
    axis=1
)
print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Last run on: 2026-05-10 20:43:59


Identify Origin

In [29]:
def identify_origin(row):
    analysis_col = row['FinalFormat']
    ColumnFormat = row['ColumnFormat']
    DescriptionFormat = row['DescriptionFormat']

    # Check if the value in 'FinalFormat' came from 'ColumnFormat'
    if pd.notnull(ColumnFormat) and analysis_col == ColumnFormat:
        return row['ColumnKeyword']

    # Check if the value in 'FinalFormat' came from 'DescriptionFormat'
    elif pd.notnull(DescriptionFormat) and analysis_col == DescriptionFormat:
        return row['DescriptionKeyword']

    # If none of the above conditions are met, return NaN or any default value you prefer
    else:
        return 'NaN'

# Create the new column 'SourceKeyword' in analysed_columns_df
analysed_columns_df['SourceKeyword'] = analysed_columns_df.apply(identify_origin, axis=1)
print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Last run on: 2026-05-10 20:44:00


Save results

In [30]:
# Save the results to a new Excel file
analysed_columns_df.to_excel(analysed_columns_file_path, index=False)

from datetime import datetime
print(f"Last run on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Last run on: 2026-05-10 20:44:29
